<a href="https://colab.research.google.com/github/Nubiga-lima/kyc-risk-model/blob/main/notebooks/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## KYC Risk Model — Feature Engineering


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import os

### 1. Load Synthetic Dataset


In [3]:
df = pd.read_csv("data/processed/synthetic_kyc_data.csv")
print("Dataset loaded. Shape:", df.shape)
df.head()

Dataset loaded. Shape: (5000, 12)


,customer_id,age,country_risk,monthly_txn_count,monthly_txn_volume,unusual_txn_flag,product_risk_score,pep_flag,adverse_media_flag,high_risk_jurisdiction,risk_score,risk_class
0,1,69,Low,23,2273.60,0,1,0,0,0,0.0,Low
1,2,32,Low,25,1570.80,0,1,0,0,0,0.0,Low
2,3,78,Low,15,539.06,0,1,0,0,0,0.0,Low
3,4,38,Low,14,1977.78,0,1,0,0,0,0.0,Low
4,5,41,Medium,20,1026.26,0,2,0,0,0,0.0,Low


### 2. Basic Data Quality Checks

In [4]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate customer IDs:", df["customer_id"].duplicated().sum())
df.describe(include="all")

Missing values:
 customer_id               0
age                       0
country_risk              0
monthly_txn_count         0
monthly_txn_volume        0
unusual_txn_flag          0
product_risk_score        0
pep_flag                  0
adverse_media_flag        0
high_risk_jurisdiction    0
risk_score                0
risk_class                0
dtype: int64

Duplicate customer IDs: 0


,customer_id,age,country_risk,monthly_txn_count,monthly_txn_volume,unusual_txn_flag,product_risk_score,pep_flag,adverse_media_flag,high_risk_jurisdiction,risk_score,risk_class
count,5000.000000,5000.000000,5000,5000.000000,5000.000000,5000.0,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000
unique,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,NaN,NaN,Low,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low
freq,NaN,NaN,3824,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4684
mean,2500.500000,51.014600,NaN,20.110600,1004.331822,0.0,1.393400,0.020400,0.048000,0.044000,0.033320,NaN
std,1443.520003,19.310613,NaN,4.481025,708.079685,0.0,0.659942,0.141378,0.213788,0.205116,0.102254,NaN
min,1.000000,18.000000,NaN,6.000000,2.760000,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,1250.750000,34.000000,NaN,17.000000,479.132500,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,NaN
50%,2500.500000,51.000000,NaN,20.000000,844.690000,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,NaN
75%,3750.250000,68.000000,NaN,23.000000,1351.282500,0.0,2.000000,0.000000,0.000000,0.000000,0.000000,NaN


### 3. Encode Categorical Variables

In [5]:
categorical_cols = ["country_risk"]

encoder = OneHotEncoder(drop="first", sparse_output=False)
encoded = encoder.fit_transform(df[categorical_cols])
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(categorical_cols)
)

print("Encoded columns generated:", encoded_df.columns.tolist())

df = pd.concat([df, encoded_df], axis=1)
df.drop(columns=categorical_cols, inplace=True)
df.head()

Encoded columns generated: ['country_risk_Low', 'country_risk_Medium']


,customer_id,age,monthly_txn_count,monthly_txn_volume,unusual_txn_flag,product_risk_score,pep_flag,adverse_media_flag,high_risk_jurisdiction,risk_score,risk_class,country_risk_Low,country_risk_Medium
0,1,69,23,2273.60,0,1,0,0,0,0.0,Low,1.0,0.0
1,2,32,25,1570.80,0,1,0,0,0,0.0,Low,1.0,0.0
2,3,78,15,539.06,0,1,0,0,0,0.0,Low,1.0,0.0
3,4,38,14,1977.78,0,1,0,0,0,0.0,Low,1.0,0.0
4,5,41,20,1026.26,0,2,0,0,0,0.0,Low,0.0,1.0


### 4. Engineered Features (KYC/FC Domain Logic)

In [6]:
# Transaction intensity: average value per transaction
df["txn_intensity"] = df["monthly_txn_volume"] / (df["monthly_txn_count"] + 1)

# High-volume flag (synthetic threshold)
df["high_volume_flag"] = (df["monthly_txn_volume"] > 5000).astype(int)

# Interaction: PEP in high-risk jurisdiction
df["pep_high_risk_interaction"] = df["pep_flag"] * df["high_risk_jurisdiction"]

# Combined adverse media + PEP risk
df["media_pep_combo"] = df["adverse_media_flag"] + df["pep_flag"]

# Behavioural risk score
df["behavioural_risk_score"] = (
    df["unusual_txn_flag"] +
    (df["monthly_txn_count"] > 50).astype(int)
)

df.head()

,customer_id,age,monthly_txn_count,monthly_txn_volume,unusual_txn_flag,product_risk_score,pep_flag,adverse_media_flag,high_risk_jurisdiction,risk_score,risk_class,country_risk_Low,country_risk_Medium,txn_intensity,high_volume_flag,pep_high_risk_interaction,media_pep_combo,behavioural_risk_score
0,1,69,23,2273.60,0,1,0,0,0,0.0,Low,1.0,0.0,94.733333,0,0,0,0
1,2,32,25,1570.80,0,1,0,0,0,0.0,Low,1.0,0.0,60.415385,0,0,0,0
2,3,78,15,539.06,0,1,0,0,0,0.0,Low,1.0,0.0,33.691250,0,0,0,0
3,4,38,14,1977.78,0,1,0,0,0,0.0,Low,1.0,0.0,131.852000,0,0,0,0
4,5,41,20,1026.26,0,2,0,0,0,0.0,Low,0.0,1.0,48.869524,0,0,0,0


### 5. Scale Numerical Variables

In [7]:
scaler = StandardScaler()
scaled_cols = [
    "age",
    "monthly_txn_count",
    "monthly_txn_volume",
    "txn_intensity",
    "behavioural_risk_score"
]

df_scaled = df.copy()
df_scaled[scaled_cols] = scaler.fit_transform(df[scaled_cols])
df_scaled.head()

,customer_id,age,monthly_txn_count,monthly_txn_volume,unusual_txn_flag,product_risk_score,pep_flag,adverse_media_flag,high_risk_jurisdiction,risk_score,risk_class,country_risk_Low,country_risk_Medium,txn_intensity,high_volume_flag,pep_high_risk_interaction,media_pep_combo,behavioural_risk_score
0,1,0.931467,0.644872,1.792729,0,1,0,0,0,0.0,Low,1.0,0.0,1.169628,0,0,0,0.0
1,2,-0.984769,1.091243,0.800086,0,1,0,0,0,0.0,Low,1.0,0.0,0.271993,0,0,0,0.0
2,3,1.397579,-1.140612,-0.657155,0,1,0,0,0,0.0,Low,1.0,0.0,-0.427014,0,0,0,0.0
3,4,-0.674028,-1.363798,1.374910,0,1,0,0,0,0.0,Low,1.0,0.0,2.140519,0,0,0,0.0
4,5,-0.518658,-0.024684,0.030972,0,2,0,0,0,0.0,Low,0.0,1.0,-0.030005,0,0,0,0.0


### 6. Final Feature Matrix

In [8]:
feature_cols = [
    "age",
    "monthly_txn_count",
    "monthly_txn_volume",
    "product_risk_score",
    "unusual_txn_flag",
    "pep_flag",
    "adverse_media_flag",
    "high_risk_jurisdiction",
    "country_risk_Low",
    "country_risk_Medium",
    "txn_intensity",
    "high_volume_flag",
    "pep_high_risk_interaction",
    "media_pep_combo",
    "behavioural_risk_score"
]

X = df_scaled[feature_cols]
y = df_scaled["risk_class"]

print("Feature matrix shape:", X.shape)
X.head()

Feature matrix shape: (5000, 15)


,age,monthly_txn_count,monthly_txn_volume,product_risk_score,unusual_txn_flag,pep_flag,adverse_media_flag,high_risk_jurisdiction,country_risk_Low,country_risk_Medium,txn_intensity,high_volume_flag,pep_high_risk_interaction,media_pep_combo,behavioural_risk_score
0,0.931467,0.644872,1.792729,1,0,0,0,0,1.0,0.0,1.169628,0,0,0,0.0
1,-0.984769,1.091243,0.800086,1,0,0,0,0,1.0,0.0,0.271993,0,0,0,0.0
2,1.397579,-1.140612,-0.657155,1,0,0,0,0,1.0,0.0,-0.427014,0,0,0,0.0
3,-0.674028,-1.363798,1.374910,1,0,0,0,0,1.0,0.0,2.140519,0,0,0,0.0
4,-0.518658,-0.024684,0.030972,2,0,0,0,0,0.0,1.0,-0.030005,0,0,0,0.0


### 7. Save Engineered Dataset

In [9]:
output_path = "data/processed/kyc_features.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_scaled.to_csv(output_path, index=False)
print(f"Saved engineered dataset to: {output_path}")

Saved engineered dataset to: data/processed/kyc_features.csv
